# LeetCode 3200 — Maximum Height of a Triangle bằng Python

Link đề: https://leetcode.com/problems/maximum-height-of-a-triangle/ · Độ khó: **Easy** · Chủ đề: Array, Enumeration (mô phỏng, tổng cấp số cộng)

Skeleton LeetCode:

```python
class Solution:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc (đề gốc có kèm hình minh họa); mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `red` bóng đỏ và `blue` bóng xanh; xếp thành tam giác **hàng 1 có 1 bóng, hàng 2 có 2 bóng, hàng 3 có 3 bóng, ...**; **mỗi hàng cùng một màu**, **hai hàng liền kề khác màu**; tìm **chiều cao lớn nhất**.
2. Nắm các công cụ Python/toán: vòng lặp `while` với điều kiện "còn đủ bóng", list hai phần tử `[red, blue]` được truy cập bằng chỉ số `0/1`, đổi màu bằng `j = 1 - j` hoặc `j ^= 1`, `max(...)`, và các tổng `1 + 2 + ... + h`, `1 + 3 + 5 + ...`, `2 + 4 + 6 + ...`.
3. Đi từ **thử từng chiều cao và tính bóng cần dùng** → **mô phỏng xếp từng hàng, trừ dần bóng** → **công thức đóng cho một chiều cao** → **tìm kiếm nhị phân trên chiều cao** → **bản gọn `j ^= 1`**.
4. Hiểu mấu chốt: màu hàng 1 **quyết định toàn bộ** cách tô (vì hàng kề nhau khác màu và chỉ có hai màu), nên chỉ có **hai** khả năng — hàng 1 đỏ hoặc hàng 1 xanh — rồi lấy chiều cao lớn hơn.
5. Tránh các lỗi kinh điển: chỉ thử một màu đầu, so sánh `<` thay vì `<=`, gộp hai màu thành một kho bóng chung, và lệch một đơn vị khi trả chiều cao.

## 2. Đề bài & ví dụ

Cho hai số nguyên `red` và `blue` là số bóng đỏ và số bóng xanh. Bạn phải xếp các bóng thành một **tam giác** sao cho **hàng thứ 1 có 1 bóng, hàng thứ 2 có 2 bóng, hàng thứ 3 có 3 bóng**, và cứ thế.

- Tất cả bóng trong **cùng một hàng** phải **cùng màu**.
- Hai hàng **liền kề** phải **khác màu**.

Hãy trả về **chiều cao lớn nhất** (số hàng nhiều nhất) mà tam giác có thể đạt được. Không bắt buộc phải dùng hết bóng.

**Ví dụ 1**

- Input: `red = 2`, `blue = 4`
- Output: `3`
- Giải thích: xếp hàng 1 xanh (1 bóng), hàng 2 đỏ (2 bóng), hàng 3 xanh (3 bóng). Dùng `1 + 3 = 4` bóng xanh và `2` bóng đỏ. Đây là cách xếp duy nhất đạt chiều cao 3.

**Ví dụ 2**

- Input: `red = 2`, `blue = 1`
- Output: `2`
- Giải thích: hàng 1 xanh (1 bóng), hàng 2 đỏ (2 bóng). Hàng 3 cần 3 bóng xanh nhưng đã hết.

**Ví dụ 3**

- Input: `red = 1`, `blue = 1`
- Output: `1`
- Giải thích: hàng 1 dùng 1 bóng; hàng 2 cần 2 bóng cùng màu mà mỗi màu chỉ còn tối đa 1 hoặc 0 bóng.

**Ví dụ 4**

- Input: `red = 10`, `blue = 1`
- Output: `2`
- Giải thích: hàng 1 xanh (1 bóng), hàng 2 đỏ (2 bóng). Hàng 3 lại cần 3 bóng xanh, mà xanh chỉ có 1 và đã dùng hết. Dù đỏ dư nhiều cũng không giúp được vì hai hàng đỏ không được kề nhau.

**Ràng buộc:** `1 <= red, blue <= 100`.

In [1]:
import random

examples = [
    (2, 4, 3),
    (2, 1, 2),
    (1, 1, 1),
    (10, 1, 2),
]
for red, blue, expected in examples:
    print(f"red={red}, blue={blue} -> kỳ vọng {expected}")

red=2, blue=4 -> kỳ vọng 3
red=2, blue=1 -> kỳ vọng 2
red=1, blue=1 -> kỳ vọng 1
red=10, blue=1 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Số bóng của tam giác cao `h`: `1 + 2 + ... + h`

Hàng `i` có `i` bóng nên tam giác cao `h` cần tổng cộng `1 + 2 + ... + h = h(h+1)/2` bóng (số tam giác). Ví dụ `h = 3` cần `6` bóng. Đây mới là **tổng cả hai màu**; vì hai màu nằm ở các hàng xen kẽ nên từng màu cần ít hơn nhiều.

In [2]:
for h in range(1, 7):
    print(f"h={h}: các hàng {list(range(1, h + 1))} -> tổng {sum(range(1, h + 1))} = h(h+1)/2 = {h * (h + 1) // 2}")

h=1: các hàng [1] -> tổng 1 = h(h+1)/2 = 1
h=2: các hàng [1, 2] -> tổng 3 = h(h+1)/2 = 3
h=3: các hàng [1, 2, 3] -> tổng 6 = h(h+1)/2 = 6
h=4: các hàng [1, 2, 3, 4] -> tổng 10 = h(h+1)/2 = 10
h=5: các hàng [1, 2, 3, 4, 5] -> tổng 15 = h(h+1)/2 = 15
h=6: các hàng [1, 2, 3, 4, 5, 6] -> tổng 21 = h(h+1)/2 = 21


### 3.2. Hai màu xen kẽ: hàng lẻ một màu, hàng chẵn màu kia

Nếu hàng 1 màu `X` thì hàng 2 màu `Y`, hàng 3 màu `X`, hàng 4 màu `Y`, ... Nghĩa là **hàng lẻ** (1, 3, 5, ...) cùng một màu và **hàng chẵn** (2, 4, 6, ...) cùng màu còn lại. Vì chỉ có **hai** màu, nên **chọn màu hàng 1 là chọn toàn bộ cách tô**.

In [3]:
h = 5
for first in ("đỏ", "xanh"):
    other = "xanh" if first == "đỏ" else "đỏ"
    colors = [first if row % 2 == 1 else other for row in range(1, h + 1)]
    print(f"hàng 1 {first}:", colors)

hàng 1 đỏ: ['đỏ', 'xanh', 'đỏ', 'xanh', 'đỏ']
hàng 1 xanh: ['xanh', 'đỏ', 'xanh', 'đỏ', 'xanh']


### 3.3. Tổng các hàng lẻ và các hàng chẵn

- Các hàng lẻ có `1, 3, 5, ...`: nếu có `m` hàng lẻ thì tổng là `m²` (`1 = 1²`, `1+3 = 4 = 2²`, `1+3+5 = 9 = 3²`).
- Các hàng chẵn có `2, 4, 6, ...`: nếu có `k` hàng chẵn thì tổng là `k(k+1)` (`2 = 1·2`, `2+4 = 6 = 2·3`, `2+4+6 = 12 = 3·4`).

Với chiều cao `h`: `m = (h + 1) // 2` hàng lẻ và `k = h // 2` hàng chẵn.

In [4]:
for h in range(1, 8):
    m, k = (h + 1) // 2, h // 2
    odd_rows = list(range(1, h + 1, 2))
    even_rows = list(range(2, h + 1, 2))
    print(f"h={h}: hàng lẻ {odd_rows} tổng {sum(odd_rows)} = m^2 = {m * m} | hàng chẵn {even_rows} tổng {sum(even_rows)} = k(k+1) = {k * (k + 1)}")

h=1: hàng lẻ [1] tổng 1 = m^2 = 1 | hàng chẵn [] tổng 0 = k(k+1) = 0
h=2: hàng lẻ [1] tổng 1 = m^2 = 1 | hàng chẵn [2] tổng 2 = k(k+1) = 2
h=3: hàng lẻ [1, 3] tổng 4 = m^2 = 4 | hàng chẵn [2] tổng 2 = k(k+1) = 2
h=4: hàng lẻ [1, 3] tổng 4 = m^2 = 4 | hàng chẵn [2, 4] tổng 6 = k(k+1) = 6
h=5: hàng lẻ [1, 3, 5] tổng 9 = m^2 = 9 | hàng chẵn [2, 4] tổng 6 = k(k+1) = 6
h=6: hàng lẻ [1, 3, 5] tổng 9 = m^2 = 9 | hàng chẵn [2, 4, 6] tổng 12 = k(k+1) = 12
h=7: hàng lẻ [1, 3, 5, 7] tổng 16 = m^2 = 16 | hàng chẵn [2, 4, 6] tổng 12 = k(k+1) = 12


### 3.4. List hai phần tử `[red, blue]` và đổi màu bằng `j = 1 - j`

Ta có thể gom hai kho bóng vào một list `c = [red, blue]` rồi dùng chỉ số `j` là `0` (đỏ) hoặc `1` (xanh). Muốn **đổi màu** cho hàng kế tiếp chỉ cần lật `j`: `j = 1 - j` hoặc `j ^= 1` (XOR với 1 lật bit cuối: `0 ^ 1 = 1`, `1 ^ 1 = 0`).

In [5]:
c = [2, 4]            # c[0] = số bóng đỏ, c[1] = số bóng xanh
j = 0
for step in range(5):
    print(f"bước {step}: j = {j} ({'đỏ' if j == 0 else 'xanh'}), kho = {c[j]}")
    j = 1 - j                    # lật 0 <-> 1
j = 0
flips = []
for _ in range(4):
    j ^= 1                       # lật bằng XOR
    flips.append(j)
print(flips)                     # [1, 0, 1, 0]

bước 0: j = 0 (đỏ), kho = 2
bước 1: j = 1 (xanh), kho = 4
bước 2: j = 0 (đỏ), kho = 2
bước 3: j = 1 (xanh), kho = 4
bước 4: j = 0 (đỏ), kho = 2
[1, 0, 1, 0]


### 3.5. Vòng `while` với điều kiện "còn đủ bóng" và `max`

Ta xếp từng hàng khi **kho của màu hiện tại còn ít nhất `i` bóng**. Dùng `while c[j] >= i:` để lặp và dừng ngay khi không đủ. `max(a, b)` chọn kết quả tốt hơn giữa hai khả năng "hàng 1 đỏ" và "hàng 1 xanh".

In [6]:
balls = 7
row = 1
while balls >= row:
    balls -= row
    print(f"xếp hàng {row}: còn {balls}")
    row += 1
print("không xếp được hàng", row, "vì còn", balls, "<", row)
print(max(3, 2), max(1, 1))

xếp hàng 1: còn 6
xếp hàng 2: còn 4
xếp hàng 3: còn 1
không xếp được hàng 4 vì còn 1 < 4
3 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `red = 2`, `blue = 4`. Thử hai cách chọn màu hàng 1.

**Hàng 1 xanh.** Hàng 1 xanh cần 1 (xanh còn `3`); hàng 2 đỏ cần 2 (đỏ còn `0`); hàng 3 xanh cần 3 (xanh còn `0`); hàng 4 đỏ cần 4 nhưng đỏ đã hết ⇒ dừng. **Chiều cao 3.**

**Hàng 1 đỏ.** Hàng 1 đỏ cần 1 (đỏ còn `1`); hàng 2 xanh cần 2 (xanh còn `2`); hàng 3 đỏ cần 3 nhưng đỏ chỉ còn `1` ⇒ dừng. **Chiều cao 2.**

Lấy lớn nhất ⇒ `3`. Ý chính: **thử hai khả năng** màu hàng 1 và **mô phỏng** đến khi kho màu hiện tại không đủ cho hàng kế tiếp.

In [7]:
red, blue = 2, 4
for first in ("xanh", "đỏ"):
    c = {"đỏ": red, "xanh": blue}
    color = first
    row = 1
    print("--- hàng 1 là", first)
    while c[color] >= row:
        c[color] -= row
        print(f"hàng {row} màu {color}: còn đỏ={c['đỏ']}, xanh={c['xanh']}")
        row += 1
        color = "xanh" if color == "đỏ" else "đỏ"
    print("dừng, chiều cao =", row - 1)

--- hàng 1 là xanh
hàng 1 màu xanh: còn đỏ=2, xanh=3
hàng 2 màu đỏ: còn đỏ=0, xanh=3
hàng 3 màu xanh: còn đỏ=0, xanh=0
dừng, chiều cao = 3
--- hàng 1 là đỏ
hàng 1 màu đỏ: còn đỏ=1, xanh=4
hàng 2 màu xanh: còn đỏ=1, xanh=2
dừng, chiều cao = 2


## 5. Cách 1 — Brute Force / baseline: thử từng chiều cao `h`

**Ý tưởng.** Với mỗi chiều cao `h` (thử `1, 2, 3, ...`) và mỗi màu hàng 1, **tính tổng số bóng mỗi màu cần** bằng cách cộng `row` vào kho của màu của hàng đó; nếu cả hai kho đủ thì `h` khả thi. Lấy `h` lớn nhất khả thi. Cách này không "thông minh": mỗi `h` tính lại từ hàng 1.

Vì `red + blue <= 200` mà tam giác cao `h` cần `h(h+1)/2` bóng, nên `h(h+1)/2 <= 200` ⇒ `h <= 19`; ta thử `h` đến `30` cho an toàn.

In [8]:
def max_height_bruteforce(red, blue):
    best = 0
    for h in range(1, 31):
        for first in (0, 1):                        # 0: hàng 1 đỏ, 1: hàng 1 xanh
            need = [0, 0]                           # need[0]: bóng đỏ cần, need[1]: bóng xanh cần
            for row in range(1, h + 1):
                color = (first + row - 1) % 2        # hàng xen kẽ màu
                need[color] += row
            if need[0] <= red and need[1] <= blue:
                best = max(best, h)
    return best


for red, blue, expected in examples:
    print(max_height_bruteforce(red, blue), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Gọi `H` là chiều cao tối đa có thể thử (`H ≈ √(2·(red+blue))`, thực tế `<= 19`). Mỗi `h` tốn `O(h)` ⇒ **Time `O(H²)`**, **Space `O(1)`**.

**Nút thắt:** ta tính lại các hàng `1..h` từ đầu cho mỗi `h`, trong khi tam giác cao `h+1` chỉ là tam giác cao `h` thêm **một hàng**. Nên xếp dần từng hàng và dừng ngay khi hết bóng.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Màu hàng 1 quyết định tất cả.** Chỉ có hai màu và hàng kề phải khác màu, nên cách tô là xen kẽ hoàn toàn. Chỉ có **2** kịch bản: hàng 1 đỏ hoặc hàng 1 xanh.

**Bước 2 — Mô phỏng tham lam, không cần quay lui.** Trong mỗi kịch bản màu của hàng `i` đã cố định; ta chỉ cần kiểm tra kho màu đó còn `>= i` không, trừ đi `i`, rồi sang hàng `i + 1` với màu kia. Nếu không đủ thì **không tồn tại** tam giác cao hơn trong kịch bản này (vì hàng `i` bắt buộc mang màu đó).

**Bước 3 — Lấy `max` hai kịch bản.**

Mỗi kịch bản chạy tối đa `~19` hàng ⇒ gần như `O(1)`; chính xác là `O(√(red + blue))`.

In [9]:
def height_when_first(x, y):
    """Chiều cao khi hàng 1 dùng màu có x bóng (màu kia có y bóng)."""
    balls = [x, y]
    j = 0                      # 0: màu của hàng 1 (x), 1: màu kia (y)
    row = 1
    while balls[j] >= row:
        balls[j] -= row
        j = 1 - j
        row += 1
    return row - 1


print(height_when_first(4, 2), height_when_first(2, 4))      # 3 2   (red=2, blue=4)
print(max(height_when_first(2, 4), height_when_first(4, 2)))  # 3

3 2
3


## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng từng hàng cho hai màu đầu (cách nên nộp)

**Giải thích từng đoạn.**

1. Hàm phụ `height_when_first(x, y)`: `balls = [x, y]`, `j = 0` là màu đang dùng, `row = 1`.
2. `while balls[j] >= row`: kho màu hiện tại đủ cho hàng `row` thì trừ `row`, lật `j`, tăng `row`.
3. Thoát vòng lặp nghĩa là hàng `row` **không** xếp được, vậy đã xếp `row - 1` hàng.
4. Đáp án là `max(height_when_first(red, blue), height_when_first(blue, red))`: lần đầu hàng 1 đỏ, lần sau hàng 1 xanh.

In [10]:
def max_height_simulate(red, blue):
    return max(height_when_first(red, blue), height_when_first(blue, red))


for red, blue, expected in examples:
    print(max_height_simulate(red, blue), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
2 (kỳ vọng 2 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSimulate:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        def height(x: int, y: int) -> int:
            balls = [x, y]
            j, row = 0, 1
            while balls[j] >= row:
                balls[j] -= row
                j = 1 - j
                row += 1
            return row - 1

        return max(height(red, blue), height(blue, red))


print(SolutionSimulate().maxHeightOfTriangle(2, 4))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(√(red + blue))`** (mỗi kịch bản tối đa `h` vòng với `h(h+1)/2 <= red + blue`), **Space `O(1)`**.

### Cách 3 — Công thức đóng cho một chiều cao + tăng dần `h`

**Giải thích từng đoạn.**

1. Với chiều cao `h`: `m = (h + 1) // 2` hàng lẻ cần `m²` bóng, `k = h // 2` hàng chẵn cần `k(k+1)` bóng.
2. `h` khả thi nếu `(m² <= red và k(k+1) <= blue)` **hoặc** `(m² <= blue và k(k+1) <= red)` — vế đầu là "hàng 1 đỏ", vế sau là "hàng 1 xanh".
3. Tăng `h` từ `0` đến khi `h + 1` không khả thi. Vì cần bóng không giảm khi `h` tăng nên nếu `h` khả thi thì mọi `h' < h` cũng khả thi (bỏ bớt hàng cuối).

In [12]:
def feasible(h, red, blue):
    m, k = (h + 1) // 2, h // 2                      # số hàng lẻ, hàng chẵn
    odd_need, even_need = m * m, k * (k + 1)
    return (odd_need <= red and even_need <= blue) or (odd_need <= blue and even_need <= red)


def max_height_formula(red, blue):
    h = 0
    while feasible(h + 1, red, blue):
        h += 1
    return h


for red, blue, expected in examples:
    print(max_height_formula(red, blue), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
2 (kỳ vọng 2 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionFormula:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        def feasible(h: int) -> bool:
            odd_need = ((h + 1) // 2) ** 2           # tổng các hàng lẻ 1 + 3 + 5 + ...
            even_need = (h // 2) * (h // 2 + 1)      # tổng các hàng chẵn 2 + 4 + 6 + ...
            return (odd_need <= red and even_need <= blue) or (odd_need <= blue and even_need <= red)

        h = 0
        while feasible(h + 1):
            h += 1
        return h


print(SolutionFormula().maxHeightOfTriangle(10, 1))      # 2

2


**Độ phức tạp Cách 3:** **Time `O(√(red + blue))`** (tối đa `~20` lần gọi `feasible`, mỗi lần `O(1)`), **Space `O(1)`**.

### Cách 4 — Tìm kiếm nhị phân trên chiều cao

**Giải thích từng đoạn.**

1. `feasible(h)` **đơn điệu**: đúng với `h` nhỏ, sai với `h` lớn (một khi sai thì sai mãi).
2. Tìm `h` lớn nhất khả thi trong `[0, 20]` (`h = 20` cần `210 > 200` bóng nên chắc chắn không khả thi; `h = 0` luôn khả thi).
3. `mid = (lo + hi + 1) // 2` (lấy trung điểm **làm tròn lên** để vòng lặp luôn tiến): nếu `feasible(mid)` thì `lo = mid`, ngược lại `hi = mid - 1`.

Với phạm vi nhỏ thế này nhị phân không nhanh hơn rõ rệt, nhưng đây là khuôn mẫu **"nhị phân trên đáp án"** rất đáng nhớ.

In [14]:
def max_height_binary_search(red, blue):
    lo, hi = 0, 20                      # feasible(0) đúng; feasible(20) sai vì cần 210 > 200 bóng
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(mid, red, blue):
            lo = mid                    # mid khả thi: đáp án >= mid
        else:
            hi = mid - 1                # mid không khả thi: đáp án < mid
    return lo


for red, blue, expected in examples:
    print(max_height_binary_search(red, blue), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
2 (kỳ vọng 2 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionBinarySearch:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        def feasible(h: int) -> bool:
            odd_need = ((h + 1) // 2) ** 2
            even_need = (h // 2) * (h // 2 + 1)
            return (odd_need <= red and even_need <= blue) or (odd_need <= blue and even_need <= red)

        lo, hi = 0, 20
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if feasible(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo


print(SolutionBinarySearch().maxHeightOfTriangle(2, 4))      # 3

3


**Độ phức tạp Cách 4:** **Time `O(log H)`** với `H = 20` (khoảng `5` lần gọi `feasible`), **Space `O(1)`**.

### Cách 5 — Bản gọn `c = [red, blue]` và `j ^= 1`

**Giải thích từng đoạn.**

1. Với `k` là `0` hoặc `1` (màu của hàng 1), đặt `c = [red, blue]`, `i = 1` (số bóng của hàng hiện tại), `j = k` (màu hiện tại).
2. `while i <= c[j]`: trừ `i` khỏi `c[j]`, lật `j ^= 1`, cập nhật `ans = max(ans, i)`, tăng `i`.
3. Đây chính là Cách 2 viết gộp: `ans` ghi nhớ hàng lớn nhất đã xếp được trong cả hai kịch bản.

In [16]:
def max_height_xor(red, blue):
    ans = 0
    for k in range(2):
        c = [red, blue]
        i, j = 1, k
        while i <= c[j]:
            c[j] -= i
            j ^= 1
            ans = max(ans, i)
            i += 1
    return ans


for red, blue, expected in examples:
    print(max_height_xor(red, blue), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )
2 (kỳ vọng 2 )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionXor:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        ans = 0
        for k in range(2):
            c = [red, blue]
            i, j = 1, k
            while i <= c[j]:
                c[j] -= i
                j ^= 1
                ans = max(ans, i)
                i += 1
        return ans


print(SolutionXor().maxHeightOfTriangle(1, 1))      # 1

1


**Độ phức tạp Cách 5:** **Time `O(√(red + blue))`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 trên `red = 2`, `blue = 4` cho **cả hai kịch bản**, và trên `red = 10`, `blue = 1`.

Kịch bản "hàng 1 xanh" (`x = 4` là xanh, `y = 2` là đỏ):

| row | `j` (màu) | `balls[j]` trước | đủ? (`balls[j] >= row`) | `balls` sau | ghi chú |
|---|---|---|---|---|---|
| 1 | 0 (xanh) | 4 | có | `[3, 2]` | xếp hàng 1 |
| 2 | 1 (đỏ) | 2 | có | `[3, 0]` | xếp hàng 2 |
| 3 | 0 (xanh) | 3 | có | `[0, 0]` | xếp hàng 3 |
| 4 | 1 (đỏ) | 0 | không | `[0, 0]` | dừng, chiều cao `4 - 1 = 3` |

Kịch bản "hàng 1 đỏ" (`x = 2`, `y = 4`): row 1 đủ (`[1, 4]`), row 2 đủ (`[1, 2]`), row 3 cần `3` mà `balls[0] = 1` ⇒ dừng, chiều cao `2`. Lấy `max(3, 2) = 3`.

In [18]:
def trace_height(x, y, name_first):
    balls = [x, y]
    j, row = 0, 1
    print(f"--- hàng 1 là {name_first}: balls = {balls}")
    while True:
        enough = balls[j] >= row
        print(f"row={row}, j={j}, balls[j]={balls[j]}, đủ? {enough}")
        if not enough:
            break
        balls[j] -= row
        print(f"    xếp hàng {row}, balls sau = {balls}")
        j = 1 - j
        row += 1
    print(f"dừng, chiều cao = {row - 1}")
    return row - 1


def trace_max_height(red, blue):
    h1 = trace_height(red, blue, "đỏ")
    h2 = trace_height(blue, red, "xanh")
    print("max =", max(h1, h2))
    return max(h1, h2)


assert trace_max_height(2, 4) == 3

--- hàng 1 là đỏ: balls = [2, 4]
row=1, j=0, balls[j]=2, đủ? True
    xếp hàng 1, balls sau = [1, 4]
row=2, j=1, balls[j]=4, đủ? True
    xếp hàng 2, balls sau = [1, 2]
row=3, j=0, balls[j]=1, đủ? False
dừng, chiều cao = 2
--- hàng 1 là xanh: balls = [4, 2]
row=1, j=0, balls[j]=4, đủ? True
    xếp hàng 1, balls sau = [3, 2]
row=2, j=1, balls[j]=2, đủ? True
    xếp hàng 2, balls sau = [3, 0]
row=3, j=0, balls[j]=3, đủ? True
    xếp hàng 3, balls sau = [0, 0]
row=4, j=1, balls[j]=0, đủ? False
dừng, chiều cao = 3
max = 3


In [19]:
assert trace_max_height(10, 1) == 2

--- hàng 1 là đỏ: balls = [10, 1]
row=1, j=0, balls[j]=10, đủ? True
    xếp hàng 1, balls sau = [9, 1]
row=2, j=1, balls[j]=1, đủ? False
dừng, chiều cao = 1
--- hàng 1 là xanh: balls = [1, 10]
row=1, j=0, balls[j]=1, đủ? True
    xếp hàng 1, balls sau = [0, 10]
row=2, j=1, balls[j]=10, đủ? True
    xếp hàng 2, balls sau = [0, 8]
row=3, j=0, balls[j]=0, đủ? False
dừng, chiều cao = 2
max = 2


Với `red = 10`, `blue = 1`: kịch bản hàng 1 đỏ chỉ xếp được hàng 1 (hàng 2 cần `2` xanh mà chỉ có `1`) ⇒ `1`; kịch bản hàng 1 xanh xếp được hàng 1 (xanh) và hàng 2 (đỏ) ⇒ `2`; hàng 3 cần `3` xanh mà xanh đã hết ⇒ `max = 2`. Có `8` bóng đỏ dư không dùng được.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (mô phỏng cho hai màu đầu): rõ ràng, ngắn, `O(√(red + blue))`.

In [20]:
class Solution:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        def height(x: int, y: int) -> int:
            balls = [x, y]            # balls[0]: kho của màu hàng hiện tại, balls[1]: kho màu kia
            j, row = 0, 1
            while balls[j] >= row:    # đủ bóng cho hàng `row`
                balls[j] -= row
                j = 1 - j             # hàng sau đổi màu
                row += 1
            return row - 1            # hàng `row` không xếp được

        return max(height(red, blue), height(blue, red))


sol = Solution()
for red, blue, expected in examples:
    print(f"red={red}, blue={blue} ->", sol.maxHeightOfTriangle(red, blue), "(kỳ vọng", expected, ")")

red=2, blue=4 -> 3 (kỳ vọng 3 )
red=2, blue=1 -> 2 (kỳ vọng 2 )
red=1, blue=1 -> 1 (kỳ vọng 1 )
red=10, blue=1 -> 2 (kỳ vọng 2 )


## 10. Vì sao thuật toán đúng

- **Chỉ hai kịch bản.** Hai hàng kề khác màu và chỉ có hai màu, nên toàn bộ màu hàng là xen kẽ, hoàn toàn xác định bởi màu hàng 1. Xét cả hai ⇒ không sót cách tô hợp lệ nào.
- **Trong một kịch bản, màu hàng `i` là cố định.** Muốn có chiều cao `>= h` thì cần xếp được **hàng 1, 2, ..., h** theo đúng màu đó, mỗi hàng `i` lấy `i` bóng từ kho màu tương ứng. Mô phỏng trừ dần chính là kiểm tra điều đó theo thứ tự `i = 1, 2, ...`.
- **Dừng sớm là đúng.** Nếu tại hàng `i` kho màu cần thiết không đủ `i` bóng thì không thể có hàng `i` (và do đó không có hàng cao hơn). Vì chiều cao là số hàng liên tiếp từ hàng 1, ta trả `i - 1`.
- **Đơn điệu (cho Cách 3 và 4).** Nếu tam giác cao `h` xếp được thì bỏ hàng cuối ta được tam giác cao `h - 1` hợp lệ, nên tập chiều cao khả thi là một đoạn `0..H*`; tìm dần hoặc nhị phân đều ra `H*`.
- **Không cần dùng hết bóng.** Đề chỉ hỏi chiều cao lớn nhất; bóng dư bị bỏ.

## 11. Phân tích Time / Space Complexity

Gọi `N = red + blue`; chiều cao đạt được `H` thỏa `H(H+1)/2 <= N`, nên `H = O(√N)` (với `N <= 200` thì `H <= 19`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng `h`, tính lại từ đầu | `O(H²)` | `O(1)` |
| 2. Mô phỏng hai màu đầu | `O(H)` | `O(1)` |
| 3. Công thức đóng + tăng dần `h` | `O(H)` | `O(1)` |
| 4. Nhị phân trên `h` | `O(log H)` | `O(1)` |
| 5. Bản gọn `j ^= 1` | `O(H)` | `O(1)` |

Với `red, blue <= 100` mọi cách đều tức thì.

## 12. Bộ test

Ta gom mọi cách vào `impls` rồi so với kỳ vọng.

In [21]:
impls = {
    "bruteforce": max_height_bruteforce,
    "simulate": max_height_simulate,
    "formula": max_height_formula,
    "binary_search": max_height_binary_search,
    "xor": max_height_xor,
    "leetcode": Solution().maxHeightOfTriangle,
    "leetcode_c2": SolutionSimulate().maxHeightOfTriangle,
    "leetcode_c3": SolutionFormula().maxHeightOfTriangle,
    "leetcode_c4": SolutionBinarySearch().maxHeightOfTriangle,
    "leetcode_c5": SolutionXor().maxHeightOfTriangle,
}


def check_all(red, blue, expected, label):
    for key, fn in impls.items():
        actual = fn(red, blue)
        assert actual == expected, (label, key, red, blue, actual, expected)
    print("PASS", label)


for idx, (red, blue, expected) in enumerate(examples, 1):
    check_all(red, blue, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
PASS ví dụ 4


### 12.1. Các trường hợp biên

In [22]:
check_all(1, 1, 1, "tối thiểu: mỗi màu 1 bóng")
check_all(1, 2, 2, "red=1, blue=2: hàng 1 đỏ, hàng 2 xanh")
check_all(2, 1, 2, "đối xứng của (1, 2)")
check_all(1, 100, 2, "một màu rất nhiều nhưng màu kia chỉ 1")
check_all(100, 1, 2, "đối xứng")
check_all(3, 3, 2, "(3,3): hàng 3 cần cùng màu với hàng 1 mà kho đã dùng 1 + 3 = 4 > 3")
check_all(4, 2, 3, "hàng 1 và 3 đỏ (1 + 3 = 4), hàng 2 xanh (2)")
check_all(2, 4, 3, "đối xứng của (4, 2)")
check_all(6, 6, 4, "1+3 = 4 và 2+4 = 6")
check_all(100, 100, max_height_bruteforce(100, 100), "kích thước tối đa")
print("h(100, 100) =", Solution().maxHeightOfTriangle(100, 100))

PASS tối thiểu: mỗi màu 1 bóng
PASS red=1, blue=2: hàng 1 đỏ, hàng 2 xanh
PASS đối xứng của (1, 2)
PASS một màu rất nhiều nhưng màu kia chỉ 1
PASS đối xứng
PASS (3,3): hàng 3 cần cùng màu với hàng 1 mà kho đã dùng 1 + 3 = 4 > 3
PASS hàng 1 và 3 đỏ (1 + 3 = 4), hàng 2 xanh (2)
PASS đối xứng của (4, 2)
PASS 1+3 = 4 và 2+4 = 6
PASS kích thước tối đa
h(100, 100) = 19


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi phổ biến (chi tiết ở mục 13): (a) chỉ thử một màu đầu; (b) dùng `<` thay vì `<=`; (c) gộp hai màu thành một kho chung; (d) lệch một đơn vị khi trả chiều cao.

In [23]:
def wrong_one_start(red, blue):
    return height_when_first(red, blue)                       # SAI: chỉ thử hàng 1 đỏ


def wrong_strict(red, blue):
    def height(x, y):
        balls = [x, y]
        j, row = 0, 1
        while balls[j] > row:                                  # SAI: phải là >=
            balls[j] -= row
            j, row = 1 - j, row + 1
        return row - 1
    return max(height(red, blue), height(blue, red))


def wrong_shared_pool(red, blue):
    total, h = red + blue, 0
    while total >= (h + 1) * (h + 2) // 2:                     # SAI: coi hai màu là một kho
        h += 1
    return h


def wrong_off_by_one(red, blue):
    def height(x, y):
        balls = [x, y]
        j, row = 0, 1
        while balls[j] >= row:
            balls[j] -= row
            j, row = 1 - j, row + 1
        return row                                              # SAI: phải là row - 1
    return max(height(red, blue), height(blue, red))


print(wrong_one_start(2, 4), "!=", Solution().maxHeightOfTriangle(2, 4))
print(wrong_strict(1, 1), "!=", Solution().maxHeightOfTriangle(1, 1))
print(wrong_shared_pool(10, 1), "!=", Solution().maxHeightOfTriangle(10, 1))
print(wrong_off_by_one(2, 4), "!=", Solution().maxHeightOfTriangle(2, 4))
assert wrong_one_start(2, 4) != 3
assert wrong_strict(1, 1) != 1
assert wrong_shared_pool(10, 1) != 2
assert wrong_off_by_one(2, 4) != 3

2 != 3
0 != 1
4 != 2
4 != 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: `1 <= red, blue <= 100`. Ta kiểm tra: (1) **ngẫu nhiên** `3000` cặp với seed cố định; (2) **vét cạn mọi cặp** `1..100 × 1..100` (`10000` cặp) so tất cả cài đặt; (3) tính chất: kết quả `<= 19` và **đối xứng** `f(r, b) = f(b, r)`, **đơn điệu** khi thêm bóng.

In [24]:
rng = random.Random(3200)
for trial in range(3000):
    red, blue = rng.randint(1, 100), rng.randint(1, 100)
    ref = max_height_bruteforce(red, blue)
    for key, fn in impls.items():
        assert fn(red, blue) == ref, (key, red, blue)
print("PASS 3000 cặp (red, blue) ngẫu nhiên")

best_seen = 0
for red in range(1, 101):
    for blue in range(1, 101):
        ref = max_height_bruteforce(red, blue)
        assert max_height_simulate(red, blue) == ref
        assert max_height_formula(red, blue) == ref
        assert max_height_binary_search(red, blue) == ref
        assert max_height_xor(red, blue) == ref
        assert Solution().maxHeightOfTriangle(red, blue) == ref
        assert ref == max_height_simulate(blue, red)                              # đối xứng
        assert ref <= Solution().maxHeightOfTriangle(red + 1, blue)               # thêm bóng không làm giảm
        best_seen = max(best_seen, ref)
print("PASS vét cạn 10000 cặp; chiều cao lớn nhất gặp được =", best_seen)
assert best_seen <= 19

PASS 3000 cặp (red, blue) ngẫu nhiên


PASS vét cạn 10000 cặp; chiều cao lớn nhất gặp được = 19


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ thử "hàng 1 đỏ"

In [25]:
def sai_mot_mau_dau(red, blue):
    balls = [red, blue]
    j, row = 0, 1
    while balls[j] >= row:
        balls[j] -= row
        j = 1 - j
        row += 1
    return row - 1                                 # SAI: không thử hàng 1 xanh


print("sai:", sai_mot_mau_dau(2, 4), "| đúng:", Solution().maxHeightOfTriangle(2, 4))
print("sai:", sai_mot_mau_dau(10, 1), "| đúng:", Solution().maxHeightOfTriangle(10, 1))

sai: 2 | đúng: 3
sai: 1 | đúng: 2


**Vì sao sai:** màu hàng 1 có thể là đỏ **hoặc** xanh và hai lựa chọn có thể cho chiều cao khác nhau (`red = 2`, `blue = 4`: `2` so với `3`). Bỏ một lựa chọn là bỏ mất khả năng tốt nhất. **Cách sửa:** gọi hàm mô phỏng hai lần (đổi chỗ hai kho) và lấy `max`.

### Lỗi 2: So sánh `<` thay vì `<=` (cần "đủ", không cần "dư")

In [26]:
def sai_so_sanh_ngat(red, blue):
    def height(x, y):
        balls = [x, y]
        j, row = 0, 1
        while balls[j] > row:                      # SAI: kho bằng đúng row vẫn đủ
            balls[j] -= row
            j = 1 - j
            row += 1
        return row - 1
    return max(height(red, blue), height(blue, red))


print("sai:", sai_so_sanh_ngat(1, 1), "| đúng:", Solution().maxHeightOfTriangle(1, 1))
print("sai:", sai_so_sanh_ngat(2, 4), "| đúng:", Solution().maxHeightOfTriangle(2, 4))

sai: 0 | đúng: 1
sai: 2 | đúng: 3


**Vì sao sai:** kho còn đúng `row` bóng vẫn **đủ** để xếp hàng đó (kho về `0`). Với `(1, 1)`, hàng 1 cần `1` bóng mà kho có `1`: `1 > 1` sai nên code dừng ngay và trả `0`. **Cách sửa:** `while balls[j] >= row`.

### Lỗi 3: Coi hai màu là một kho chung

In [27]:
def sai_kho_chung(red, blue):
    total, h = red + blue, 0
    while total >= (h + 1) * (h + 2) // 2:         # SAI: chỉ kiểm tra tổng số bóng
        h += 1
    return h


print("sai:", sai_kho_chung(10, 1), "| đúng:", Solution().maxHeightOfTriangle(10, 1))
print("sai:", sai_kho_chung(1, 100), "| đúng:", Solution().maxHeightOfTriangle(1, 100))

sai: 4 | đúng: 2
sai: 13 | đúng: 2


**Vì sao sai:** tam giác cao `h` cần `h(h+1)/2` bóng tổng, nhưng bóng phải **chia đúng cho hai màu** theo hàng lẻ/chẵn. `(10, 1)`: tổng `11` đủ cho `h = 4` (cần `10`) nhưng hàng chẵn `2 + 4 = 6` xanh mà chỉ có `1` xanh (hoặc hàng lẻ `1 + 3 = 4`). **Cách sửa:** tách từng màu: hàng lẻ `m²` và hàng chẵn `k(k+1)`.

### Lỗi 4: Trả `row` thay vì `row - 1` (lệch một đơn vị)

In [28]:
def sai_lech_mot(red, blue):
    def height(x, y):
        balls = [x, y]
        j, row = 0, 1
        while balls[j] >= row:
            balls[j] -= row
            j = 1 - j
            row += 1
        return row                                 # SAI: row là hàng KẾ TIẾP, chưa xếp được
    return max(height(red, blue), height(blue, red))


print("sai:", sai_lech_mot(2, 4), "| đúng:", Solution().maxHeightOfTriangle(2, 4))

sai: 4 | đúng: 3


**Vì sao sai:** khi thoát vòng `while`, `row` là hàng **đầu tiên không xếp được**; số hàng đã xếp là `row - 1`. **Cách sửa:** trả `row - 1`, hoặc cập nhật `ans = max(ans, row)` **ngay sau** khi xếp thành công (như Cách 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xếp từng hàng với số lượng tăng dần và điều kiện **xen kẽ**" ⇒ chọn thứ nhất của dãy, các lựa chọn sau **bị ép** ⇒ mô phỏng hai kịch bản.
- Hai lựa chọn rời rạc (bắt đầu bằng A hoặc B) ⇒ chạy cả hai, lấy `max`.
- Hàng lẻ `1 + 3 + 5 + ... = m²`; hàng chẵn `2 + 4 + 6 + ... = k(k+1)`; cả hai cộng lại `h(h+1)/2`.
- Khi bài hỏi "lớn nhất `h` sao cho điều kiện đơn điệu" ⇒ **nhị phân trên đáp án** hoặc tăng dần nếu `h` nhỏ.
- Chiều cao `h` thỏa `h(h+1)/2 <= red + blue` nên `h = O(√N)`: mô phỏng vài chục bước là đủ.
- Kiểm tra bằng hai cặp "một màu rất nhiều, một màu rất ít" (như `(10, 1)`) để thấy việc chia bóng theo màu quan trọng.

## 15. Biến thể của bài

1. **Số bóng mỗi hàng khác:** hàng `i` cần `2i - 1` bóng (`1, 3, 5, ...`).
2. **Ba màu:** hàng liền kề khác màu nhưng có ba màu; số kịch bản tăng (không còn ép hoàn toàn).
3. **Trả về cách xếp:** in danh sách màu từng hàng của một tam giác cao nhất.
4. **Số bóng dư:** tính số bóng không dùng đến sau khi xếp tam giác cao nhất.
5. Bài liên quan: **Arranging Coins (441)**, **Sqrt(x) (69)**, **Valid Perfect Square (367)**, **Distribute Candies to People (1103)**.

In [29]:
def best_colors(red, blue):
    """Trả về danh sách màu từng hàng của một tam giác cao nhất (R/B)."""
    def build(x, y, first_name, second_name):
        balls, names = [x, y], [first_name, second_name]
        j, row, rows = 0, 1, []
        while balls[j] >= row:
            balls[j] -= row
            rows.append(names[j])
            j, row = 1 - j, row + 1
        return rows

    a = build(red, blue, "R", "B")
    b = build(blue, red, "B", "R")
    return a if len(a) >= len(b) else b


def max_height_odd_rows(red, blue):
    """Biến thể: hàng i cần 2i - 1 bóng (1, 3, 5, ...)."""
    def height(x, y):
        balls, j, row = [x, y], 0, 1
        while balls[j] >= 2 * row - 1:
            balls[j] -= 2 * row - 1
            j, row = 1 - j, row + 1
        return row - 1
    return max(height(red, blue), height(blue, red))


print(best_colors(2, 4))                     # ['B', 'R', 'B']
print(best_colors(10, 1))                    # ['B', 'R']
print(max_height_odd_rows(10, 10))

['B', 'R', 'B']
['B', 'R']
4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng `h` | tính bóng cần cho mỗi `h`, mỗi màu đầu | `O(H²)` | `O(1)` | hiểu đề, kiểm chứng |
| 2. Mô phỏng hai màu đầu | trừ dần từng hàng | `O(H)` | `O(1)` | **nên nộp** |
| 3. Công thức + tăng dần | `m²` và `k(k+1)` cho mỗi `h` | `O(H)` | `O(1)` | khi muốn công thức rõ ràng |
| 4. Nhị phân trên `h` | `feasible` đơn điệu | `O(log H)` | `O(1)` | khi `h` có thể rất lớn |
| 5. `j ^= 1` gọn | Cách 2 viết gộp | `O(H)` | `O(1)` | khi cần code ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `1 + 2 + 3 + 4`, `[3, 4][1]`, `1 - 0` và `1 - 1`, `1 ^ 1`, `Solution().maxHeightOfTriangle(1, 1)`, `Solution().maxHeightOfTriangle(2, 1)`.

**Bài 2 (tính tay).** Với `red = 3`, `blue = 5`: thử hàng 1 đỏ và hàng 1 xanh, lập bảng từng hàng (kho trước, kho sau) và cho biết chiều cao.

**Bài 3 (điền TODO).** Hoàn thành `height_todo(x, y)` (chiều cao khi hàng 1 dùng màu có `x` bóng).

In [30]:
def height_todo(x, y):
    # TODO 1: balls = [x, y]; j = 0; row = 1
    # TODO 2: while balls[j] >= row: trừ row khỏi balls[j], đổi màu j, tăng row
    # TODO 3: trả về row - 1
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_height_by_sum(red, blue)` bằng **công thức hàng lẻ `m²` / hàng chẵn `k(k+1)`** (Cách 3) mà **không** dùng vòng mô phỏng từng hàng; chỉ dùng một vòng tăng `h`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `best_arrangement(red, blue)` trả về danh sách màu `'R'`/`'B'` từng hàng của một tam giác cao nhất (khi hòa, ưu tiên hàng 1 đỏ). (b) Viết `leftover_balls(red, blue)` trả về `(đỏ dư, xanh dư)` theo cách xếp ở (a).

In [31]:
def max_height_by_sum(red, blue):
    # TODO: tăng h; kiểm tra m*m, k*(k+1) cho hai kịch bản
    raise NotImplementedError


def best_arrangement(red, blue):
    # TODO: mô phỏng hai kịch bản, trả danh sách màu của kịch bản cao hơn
    raise NotImplementedError


def leftover_balls(red, blue):
    # TODO: dựa vào best_arrangement, trừ số bóng đã dùng
    raise NotImplementedError

# KHÔNG gọi các hàm TODO này khi Run All.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `10`, `4`, `1` và `0`, `0`, `1`, `2`. Bài 3: khởi tạo `balls = [x, y]`, `j = 0`, `row = 1`.
- **Mức 2 (vừa):** Bài 2: hàng 1 đỏ: row 1 (đỏ `3 -> 2`), row 2 (xanh `5 -> 3`), row 3 cần `3` đỏ mà còn `2` ⇒ cao `2`. Hàng 1 xanh: row 1 (xanh `5 -> 4`), row 2 (đỏ `3 -> 1`), row 3 (xanh `4 -> 1`), row 4 cần `4` đỏ mà còn `1` ⇒ cao `3`. Đáp án `3`. Bài 4: `m = (h + 1) // 2`, `k = h // 2`.
- **Mức 3 (sâu):** Bài 5(a): chạy hai kịch bản, giữ danh sách tên màu theo từng hàng xếp được; chọn danh sách dài hơn (hòa thì chọn đỏ). Bài 5(b): `đỏ dư = red - (tổng hàng màu R)`, `xanh dư = blue - (tổng hàng màu B)`; tổng hàng `i` là `i`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 1 + 2 + 3 + 4 == 10
assert [3, 4][1] == 4
assert (1 - 0, 1 - 1) == (1, 0)
assert 1 ^ 1 == 0
assert Solution().maxHeightOfTriangle(1, 1) == 1
assert Solution().maxHeightOfTriangle(2, 1) == 2

# Bài 2
assert height_when_first(3, 5) == 2          # hàng 1 đỏ
assert height_when_first(5, 3) == 3          # hàng 1 xanh
assert Solution().maxHeightOfTriangle(3, 5) == 3


# Bài 3
def height_todo_ans(x, y):
    balls = [x, y]
    j = 0
    row = 1
    while balls[j] >= row:
        balls[j] -= row
        j = 1 - j
        row += 1
    return row - 1


for x in range(1, 40):
    for y in range(1, 40):
        assert height_todo_ans(x, y) == height_when_first(x, y)


# Bài 4
def max_height_by_sum_ans(red, blue):
    h = 0
    while True:
        nh = h + 1
        m, k = (nh + 1) // 2, nh // 2
        odd_need, even_need = m * m, k * (k + 1)
        if (odd_need <= red and even_need <= blue) or (odd_need <= blue and even_need <= red):
            h = nh
        else:
            return h


for red in range(1, 101):
    for blue in range(1, 101):
        assert max_height_by_sum_ans(red, blue) == Solution().maxHeightOfTriangle(red, blue)


# Bài 5
def best_arrangement_ans(red, blue):
    def build(x, y, first_name, second_name):
        balls, names = [x, y], [first_name, second_name]
        j, row, rows = 0, 1, []
        while balls[j] >= row:
            balls[j] -= row
            rows.append(names[j])
            j, row = 1 - j, row + 1
        return rows

    a = build(red, blue, "R", "B")
    b = build(blue, red, "B", "R")
    return a if len(a) >= len(b) else b


def leftover_balls_ans(red, blue):
    rows = best_arrangement_ans(red, blue)
    used_r = sum(i for i, color in enumerate(rows, 1) if color == "R")
    used_b = sum(i for i, color in enumerate(rows, 1) if color == "B")
    return red - used_r, blue - used_b


assert best_arrangement_ans(2, 4) == ["B", "R", "B"]
assert best_arrangement_ans(10, 1) == ["B", "R"]
assert best_arrangement_ans(1, 1) == ["R"]
assert leftover_balls_ans(2, 4) == (0, 0)
assert leftover_balls_ans(10, 1) == (8, 0)
for red in range(1, 60):
    for blue in range(1, 60):
        rows = best_arrangement_ans(red, blue)
        assert len(rows) == Solution().maxHeightOfTriangle(red, blue)
        assert all(rows[i] != rows[i + 1] for i in range(len(rows) - 1))
        lr, lb = leftover_balls_ans(red, blue)
        assert lr >= 0 and lb >= 0
print("Tất cả đáp án bài tập đều PASS")

Tất cả đáp án bài tập đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai kịch bản màu hàng 1, mô phỏng trừ dần, lấy `max`)

```python
class Solution:
    def maxHeightOfTriangle(self, red: int, blue: int) -> int:
        def height(x: int, y: int) -> int:
            balls = [x, y]
            j, row = 0, 1
            while balls[j] >= row:
                balls[j] -= row
                j = 1 - j
                row += 1
            return row - 1

        return max(height(red, blue), height(blue, red))
```

**Checklist**

- [ ] Đã thử **cả hai** màu cho hàng 1 (đổi chỗ `red` và `blue`)?
- [ ] Điều kiện xếp hàng là `balls[j] >= row` (không phải `>`)?
- [ ] Mỗi hàng chỉ lấy bóng từ **một** màu và hàng sau **đổi màu**?
- [ ] Trả `row - 1` (hàng đã xếp được), không phải `row`?
- [ ] Không gộp hai màu thành một kho chung?
- [ ] Đã thử `(2, 4) = 3`, `(2, 1) = 2`, `(1, 1) = 1`, `(10, 1) = 2`?